<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

## **Proyecto Sistemas de Recomendación**

**Autor:** Anais Aponte  
**Bootcamp:** 4Geeks Academy – Intro to Machine Learning  
**Proyecto:** Tu Futuro según la Data 

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Instrucciones**

**Objetivo:** construir un modelo de clasificación supervisada que, a partir de datos demográficos y socioeconómicos de una persona adulta (edad, nivel educativo, ocupación, estado civil, país de origen, etc.), prediga si dicha persona ganará más o menos de 50,000 USD al año.

En base a los resultados del modelo, se deberá desarrollar un sistema de recomendación interpretativo, capaz de sugerir posibles estrategias o cambios para aumentar la probabilidad de superar ese umbral de ingresos.

</div>

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 1: Carga del conjunto de datos**

El dataset utilizado en este proyecto es **adult-census-income.csv**.

Para facilitar la reproducibilidad y evitar dependencias externas, el archivo ha sido previamente descargado y almacenado dentro del repositorio en la siguiente ruta:

📁 `data/raw/adult-census-income.csv`

De esta forma, el notebook puede ejecutarse directamente sin necesidad de acceder a enlaces externos.

A continuación se realizan primero las importaciones necesarias y, posteriormente, la carga del dataset para iniciar el análisis.

</div>

In [1]:
# IMPORTACIONES

# Manejo de datos
import pandas as pd
import numpy as np

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Cargamos el fichero con los datos a analizar desde la carpeta raw del repositorio
# Usamos ruta relativa para asegurar portabilidad del notebook
df = pd.read_csv('../data/raw/adult-census-income.csv')

# Visualizamos una muestra aleatoria de 10 registros para tener una primera impresión del dataset
# y evitar sesgarnos por un posible orden previo de las filas (ya que algunos datasets pueden estar ordenados (por fecha, id, etc.)
df.sample(10)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
16252,30,Private,101859,7th-8th,4,Never-married,Craft-repair,Not-in-family,White,Male,0,0,50,United-States,<=50K
26773,66,Self-emp-inc,179951,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,70,United-States,<=50K
15465,51,Private,115025,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
1381,34,Self-emp-not-inc,156809,Some-college,10,Never-married,Craft-repair,Not-in-family,White,Male,0,1504,60,United-States,<=50K
31992,62,?,31577,HS-grad,9,Married-civ-spouse,?,Husband,White,Male,0,0,18,United-States,<=50K
22853,35,Self-emp-not-inc,108198,HS-grad,9,Divorced,Craft-repair,Own-child,Amer-Indian-Eskimo,Male,0,0,15,United-States,<=50K
9381,41,Private,63596,Some-college,10,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,32,United-States,>50K
9397,24,Private,301199,Some-college,10,Never-married,Tech-support,Own-child,White,Female,0,0,20,United-States,<=50K
7408,18,Private,107277,11th,7,Never-married,Sales,Own-child,White,Female,0,0,20,United-States,<=50K
18002,27,Private,110931,HS-grad,9,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,<=50K


-----

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 2: Preprocesamiento de datos**

Haz la limpieza de datos nulos o mal codificados, la transformación de variables categóricas, normaliza las variables numéricas.

</div>

In [3]:
# Dimensiones
df.shape

(32561, 15)

In [4]:
# Valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [5]:
# Recorremos todas las variables categóricas (tipo texto) para analizar sus valores únicos y su frecuencia
for col in df.select_dtypes(include=['object', 'string']).columns:
    print(f"\n🔹 {col}")
    print(df[col].value_counts())


🔹 workclass
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

🔹 education
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

🔹 marital.status
marital.status
Married-civ-spouse       14976
Never-married            10683
Divorced                  4443
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

🔹 occupation
occupation
Prof-specialty 

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Observación**

Se identificaron valores desconocidos representados como “?” en variables categóricas como *workclass*, *occupation* y *native.country*.  
Dado que estos casos no son despreciables y contienen información implícita, se decide tratarlos como una categoría adicional (“Unknown”) en lugar de eliminarlos o imputarlos, evitando así la pérdida de datos y permitiendo que el modelo capture su posible impacto.


</div>

In [6]:
# Reemplazamos los valores desconocidos ("?") por la categoría "Unknown" en variables categóricas
df['workclass'] = df['workclass'].replace('?', 'Unknown')
df['occupation'] = df['occupation'].replace('?', 'Unknown')
df['native.country'] = df['native.country'].replace('?', 'Unknown')

In [7]:
# Eliminamos la variable education porque ya existe su equivalente numérico education.num
df = df.drop(columns=['education'])

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Observación**

La variable education fue eliminada al existir su equivalente numérico (education.num), evitando así redundancia en el modelo.
El resto de variables categóricas serán transformadas mediante One-Hot Encoding al no presentar un orden natural.


</div>

In [8]:
# Identificamos las variables categóricas (tipo texto) que vamos a codificar
cat_cols = df.select_dtypes(include=['object', 'string']).columns
cat_cols

Index(['workclass', 'marital.status', 'occupation', 'relationship', 'race',
       'sex', 'native.country', 'income'],
      dtype='str')

In [9]:
# Aplicamos One-Hot Encoding a las variables categóricas creando variables dummy
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Observación**

Se aplicó One-Hot Encoding y se utilizó `drop_first=True` para evitar multicolinealidad, eliminando la redundancia entre variables sin pérdida de información.

</div>

In [10]:
# Visualizamos el nuevo dataset
df_encoded.sample(10)

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Unknown,native.country_Vietnam,native.country_Yugoslavia,income_>50K
6009,46,29696,9,0,0,40,False,False,True,False,...,False,False,False,False,False,True,False,False,False,True
6414,27,466224,10,0,0,40,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
2052,56,109856,13,15024,0,50,False,False,False,True,...,False,False,False,False,False,True,False,False,False,True
24600,17,806316,7,0,0,20,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
5262,24,191269,13,0,0,65,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
29927,38,194304,10,0,0,55,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
7855,30,98733,10,0,0,20,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
29803,66,178120,3,0,0,15,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
19401,38,243484,13,0,0,28,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True
13146,27,192283,11,0,0,38,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False


<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Observación**

Para el escalado de las variables numericas se optó por utilizar StandardScaler en lugar de MinMaxScaler debido a la presencia de variables con valores extremos (outliers), ya que la estandarización es menos sensible a estos y preserva mejor la distribución de los datos.

</div>

In [11]:
# Identificamos las variables numéricas
num_cols = df_encoded.select_dtypes(include=['int64', 'float64']).columns

In [12]:
# Inicializamos el escalador
scaler = StandardScaler()

# Aplicamos la normalización a las variables numéricas
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

In [13]:
# Visualizamos de nuevo dataset
df_encoded.sample(10)

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Unknown,native.country_Vietnam,native.country_Yugoslavia,income_>50K
13699,-0.262580,-1.505115,-0.420060,-0.145920,-0.216660,0.774468,False,False,True,False,...,False,False,False,False,False,True,False,False,False,True
25731,-0.409205,-0.106031,-0.031360,-0.145920,-0.216660,-0.278399,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
17790,-0.702455,0.332356,-0.031360,-0.145920,-0.216660,1.584366,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1271,-0.922393,-0.379394,-0.420060,-0.145920,3.739127,-0.035429,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
24020,-0.775768,-0.075817,-0.031360,-0.145920,-0.216660,-0.035429,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
13083,-0.702455,-0.109280,-0.420060,-0.145920,-0.216660,-0.035429,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
6563,-1.508894,-0.541234,-1.197459,-0.145920,-0.216660,-1.655225,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
27328,-0.775768,-0.028976,-0.031360,-0.145920,-0.216660,0.774468,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
3303,0.910422,-0.626020,-0.420060,0.447972,-0.216660,-0.035429,False,False,True,False,...,False,False,False,False,False,True,False,False,False,True
16288,2.449986,-0.951975,-0.420060,-0.145920,-0.216660,-1.979184,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False


---------

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 3: Define el problema de recomendación.**

Plantea cómo vas a estructurar tu sistema de recomendación:
- ¿Qué se quiere recomendar?
- ¿Cuál será el "usuario" en este caso?
- ¿Qué variables definen el perfil de un usuario?


</div>

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

##### **¿Qué se quiere recomendar?**

Las recomendaciones se enfocan en usuarios cuyo nivel de ingresos es menor o igual a 50K anuales, con el objetivo de ayudarles a mejorar su situación económica.

El sistema no se basará en la simulación de probabilidades, sino en un enfoque de similitud entre usuarios, donde se comparan perfiles a partir de sus características para identificar patrones comunes.

A partir de esta comparación, se propondrán posibles cambios en variables clave como el nivel educativo, la ocupación o el número de horas trabajadas, tomando como referencia patrones observados en usuarios con mayor nivel de ingresos.

De esta forma, las recomendaciones se basan en evidencia real del dataset y no en suposiciones teóricas.

</div>

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

##### **¿Cuál será el "usuario" en este caso?**

El usuario corresponde a cada individuo representado en el dataset, es decir, cada fila que contiene sus características demográficas y socioeconómicas.

Sobre este usuario se aplicará el sistema de recomendación, comparándolo con otros perfiles para generar sugerencias orientadas a mejorar su nivel de ingresos.

</div>

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

##### **¿Qué variables definen el perfil de un usuario?**

El perfil del usuario se define a partir de variables demográficas y socioeconómicas que influyen en su nivel de ingresos, como el tipo de empleo (*workclass*), el nivel educativo (*education.num*), la ocupación (*occupation*) y el número de horas trabajadas por semana (*hours.per.week*).

Aunque existen otras variables en el dataset, se priorizan aquellas que son interpretables y potencialmente modificables, ya que permiten generar recomendaciones más realistas y accionables para el usuario.

</div>

----

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 4: Construye el sistema de recomendación**: 

#### Enfoque filtrado basado en contenido

Representa a cada usuario como un vector y calcula similitudes entre usuarios y recomendaciones.

</div>

In [14]:
# Usuarios objetivo (<=50K)
df_low = df_encoded[df_encoded['income_>50K'] == 0]

# Usuarios referencia (>50K)
df_high = df_encoded[df_encoded['income_>50K'] == 1]

In [15]:
# Eliminamos la variable objetivo para calcular similitud
X_low = df_low.drop(columns=['income_>50K'])
X_high = df_high.drop(columns=['income_>50K'])

In [16]:
# Calculamos similitud entre usuarios
similarity_matrix = cosine_similarity(X_low, X_high)

In [17]:
# Para cada usuario bajo, encontramos el más similar alto
closest_indices = np.argmax(similarity_matrix, axis=1)

In [18]:
# Seleccionamos algunos ejemplos para inspeccionar
n_examples = 3

for i in range(n_examples):
    print(f"\n🔹 Usuario <=50K (índice {i})")
    display(X_low.iloc[i])

    print(f"\n🔹 Usuario similar >50K")
    display(X_high.iloc[closest_indices[i]])

    print("\n" + "-"*50)


🔹 Usuario <=50K (índice 0)


age                                3.769612
fnlwgt                            -1.067997
education.num                      -0.42006
capital.gain                       -0.14592
capital.loss                      10.593507
                                    ...    
native.country_Trinadad&Tobago        False
native.country_United-States           True
native.country_Unknown                False
native.country_Vietnam                False
native.country_Yugoslavia             False
Name: 0, Length: 85, dtype: object


🔹 Usuario similar >50K


age                               1.350297
fnlwgt                           -0.648199
education.num                     -0.03136
capital.gain                      -0.14592
capital.loss                      5.386957
                                    ...   
native.country_Trinadad&Tobago       False
native.country_United-States          True
native.country_Unknown               False
native.country_Vietnam               False
native.country_Yugoslavia            False
Name: 180, Length: 85, dtype: object


--------------------------------------------------

🔹 Usuario <=50K (índice 1)


...


--------------------------------------------------

🔹 Usuario <=50K (índice 2)


age                                 2.01011
fnlwgt                             -0.03522
education.num                      -0.03136
capital.gain                       -0.14592
capital.loss                      10.593507
                                    ...    
native.country_Trinadad&Tobago        False
native.country_United-States           True
native.country_Unknown                False
native.country_Vietnam                False
native.country_Yugoslavia             False
Name: 2, Length: 85, dtype: object


🔹 Usuario similar >50K


age                               0.910422
fnlwgt                            -0.70785
education.num                     -0.03136
capital.gain                      -0.14592
capital.loss                      6.104161
                                    ...   
native.country_Trinadad&Tobago       False
native.country_United-States          True
native.country_Unknown               False
native.country_Vietnam               False
native.country_Yugoslavia            False
Name: 42, Length: 85, dtype: object


--------------------------------------------------


In [21]:
# Generamos recomendaciones de negocio comparando el perfil hipotético con usuarios similares >50K
def generar_recomendaciones(perfil_original, usuarios_similares_originales):
    recomendaciones = []

    # Recomendación por nivel educativo
    educacion_referencia = usuarios_similares_originales['education.num'].median()
    if perfil_original['education.num'] < educacion_referencia:
        recomendaciones.append(
            f"Aumentar el nivel educativo: perfiles similares con ingresos >50K tienen una mediana de education.num = {educacion_referencia:.0f}, frente a {perfil_original['education.num']} en este perfil."
        )

    # Recomendación por horas trabajadas
    horas_referencia = usuarios_similares_originales['hours.per.week'].median()
    if perfil_original['hours.per.week'] < horas_referencia:
        recomendaciones.append(
            f"Aumentar las horas trabajadas por semana: perfiles similares con ingresos >50K trabajan una mediana de {horas_referencia:.0f} horas/semana, frente a {perfil_original['hours.per.week']}."
        )

    # Recomendación por ocupación
    ocupacion_referencia = usuarios_similares_originales['occupation'].mode()[0]
    if perfil_original['occupation'] != ocupacion_referencia:
        recomendaciones.append(
            f"Explorar trayectorias hacia ocupaciones como '{ocupacion_referencia}', frecuente entre perfiles similares con ingresos >50K."
        )

    return recomendaciones

In [25]:
# Buscamos un usuario real con ingresos <=50K que genere al menos una recomendación
for usuario_index in df_low.index:
    
    # Seleccionamos el usuario codificado
    usuario_low_encoded = df_low.loc[[usuario_index]]
    
    # Eliminamos la variable objetivo para calcular similitud
    usuario_low_features = usuario_low_encoded.drop(columns=['income_>50K'])
    
    # Calculamos la similitud con usuarios de ingresos >50K
    similarities = cosine_similarity(usuario_low_features, X_high)
    
    # Seleccionamos los 5 usuarios >50K más similares
    top_indices = np.argsort(similarities[0])[-5:][::-1]
    
    # Recuperamos el usuario y sus similares en el dataset original
    usuario_original = df.loc[usuario_index]
    usuarios_similares_originales = df.loc[df_high.iloc[top_indices].index]
    
    # Generamos recomendaciones
    recomendaciones = generar_recomendaciones(usuario_original, usuarios_similares_originales)
    
    # Si encontramos recomendaciones, detenemos la búsqueda
    if len(recomendaciones) > 0:
        break

In [26]:
# Mostramos el usuario evaluado
print("Usuario real evaluado con ingresos <=50K:")
display(usuario_original[['age', 'workclass', 'education.num', 'occupation', 'hours.per.week', 'income']])

# Mostramos usuarios similares con ingresos >50K
print("Usuarios similares con ingresos >50K:")
display(usuarios_similares_originales[['age', 'workclass', 'education.num', 'occupation', 'hours.per.week', 'income']])

# Mostramos las recomendaciones de negocio
print("Recomendaciones:")
for i, recomendacion in enumerate(recomendaciones, start=1):
    print(f"{i}. {recomendacion}")

Usuario real evaluado con ingresos <=50K:


age                    90
workclass         Unknown
education.num           9
occupation        Unknown
hours.per.week         40
income              <=50K
Name: 0, dtype: object

Usuarios similares con ingresos >50K:


,age,workclass,education.num,occupation,hours.per.week,income
180,57,Private,10,Adm-clerical,38,>50K
42,51,Private,10,Adm-clerical,40,>50K
130,71,Private,9,Exec-managerial,45,>50K
111,67,Private,13,Exec-managerial,40,>50K
691,61,Unknown,9,Unknown,40,>50K


Recomendaciones:
1. Aumentar el nivel educativo: perfiles similares con ingresos >50K tienen una mediana de education.num = 10, frente a 9 en este perfil.
2. Explorar trayectorias hacia ocupaciones como 'Adm-clerical', frecuente entre perfiles similares con ingresos >50K.


<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Recomendacion basada en filtrado por contenido**

Para mostrar el funcionamiento del sistema, se seleccionó un usuario real con ingresos <=50K para el cual existieran diferencias relevantes frente a perfiles similares con ingresos >50K.

El sistema genera recomendaciones únicamente cuando detecta diferencias en variables accionables como nivel educativo, ocupación u horas trabajadas por semana. Por ello, en algunos casos puede no mostrar recomendaciones si el usuario evaluado ya comparte características muy similares con los perfiles de referencia.

</div>

----

<div style="background-color:#d6eaff; border-left:6px solid #4a90e2; padding:15px; border-radius:8px; color:#1b1b1b;">

### **Paso 5: Pruebas con casos simulados**

</div>

In [27]:
# Definimos dos perfiles hipotéticos para probar el sistema de recomendación
perfiles_simulados = [
    {
        'age': 25,
        'workclass': 'Private',
        'fnlwgt': 200000,
        'education.num': 9,
        'marital.status': 'Never-married',
        'occupation': 'Adm-clerical',
        'relationship': 'Not-in-family',
        'race': 'White',
        'sex': 'Female',
        'capital.gain': 0,
        'capital.loss': 0,
        'hours.per.week': 30,
        'native.country': 'United-States'
    },
    {
        'age': 38,
        'workclass': 'Private',
        'fnlwgt': 180000,
        'education.num': 9,
        'marital.status': 'Married-civ-spouse',
        'occupation': 'Other-service',
        'relationship': 'Husband',
        'race': 'White',
        'sex': 'Male',
        'capital.gain': 0,
        'capital.loss': 0,
        'hours.per.week': 35,
        'native.country': 'United-States'
    }
]

# Evaluamos cada perfil simulado con el sistema de recomendación
for idx, perfil_usuario in enumerate(perfiles_simulados, start=1):

    # Convertimos el perfil simulado en DataFrame
    perfil_df = pd.DataFrame([perfil_usuario])

    # Aplicamos One-Hot Encoding al perfil simulado
    perfil_encoded = pd.get_dummies(perfil_df, drop_first=True)

    # Alineamos las columnas del perfil con las columnas usadas en el sistema
    perfil_encoded = perfil_encoded.reindex(columns=X_high.columns, fill_value=0)

    # Aplicamos la misma estandarización a las variables numéricas
    perfil_encoded[num_cols] = scaler.transform(perfil_encoded[num_cols])

    # Calculamos la similitud entre el perfil simulado y los usuarios con ingresos >50K
    similarities = cosine_similarity(perfil_encoded, X_high)

    # Obtenemos los 5 usuarios >50K más similares al perfil simulado
    top_indices = np.argsort(similarities[0])[-5:][::-1]

    # Recuperamos los usuarios similares desde el dataset original
    usuarios_similares_originales = df.loc[df_high.iloc[top_indices].index]

    # Generamos recomendaciones concretas para el perfil simulado
    recomendaciones = generar_recomendaciones(perfil_usuario, usuarios_similares_originales)

    # Mostramos el perfil simulado evaluado
    print(f"\n🔹 Perfil simulado {idx}")
    display(pd.DataFrame([perfil_usuario]))

    # Mostramos los usuarios similares con ingresos >50K
    print("Usuarios similares con ingresos >50K:")
    display(usuarios_similares_originales[['age', 'workclass', 'education.num', 'occupation', 'hours.per.week', 'income']])

    # Mostramos las recomendaciones generadas
    print("Recomendaciones:")
    if recomendaciones:
        for i, recomendacion in enumerate(recomendaciones, start=1):
            print(f"{i}. {recomendacion}")
    else:
        print("No se identificaron recomendaciones claras para este perfil.")

    print("\n" + "-"*80)


🔹 Perfil simulado 1


,age,workclass,fnlwgt,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,25,Private,200000,9,Never-married,Adm-clerical,Not-in-family,White,Female,0,0,30,United-States


Usuarios similares con ingresos >50K:


,age,workclass,education.num,occupation,hours.per.week,income
8702,32,Unknown,9,Unknown,2,>50K
32170,29,Private,9,Prof-specialty,10,>50K
23321,30,Private,10,Prof-specialty,5,>50K
20489,30,Self-emp-not-inc,10,Adm-clerical,20,>50K
8804,31,Self-emp-inc,10,Adm-clerical,20,>50K


Recomendaciones:
1. Aumentar el nivel educativo: perfiles similares con ingresos >50K tienen una mediana de education.num = 10, frente a 9 en este perfil.

--------------------------------------------------------------------------------

🔹 Perfil simulado 2


,age,workclass,fnlwgt,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,38,Private,180000,9,Married-civ-spouse,Other-service,Husband,White,Male,0,0,35,United-States


Usuarios similares con ingresos >50K:


,age,workclass,education.num,occupation,hours.per.week,income
8702,32,Unknown,9,Unknown,2,>50K
32170,29,Private,9,Prof-specialty,10,>50K
6145,60,Unknown,3,Unknown,30,>50K
19390,47,Local-gov,9,Other-service,7,>50K
29460,40,Federal-gov,9,Adm-clerical,20,>50K


Recomendaciones:
1. Explorar trayectorias hacia ocupaciones como 'Unknown', frecuente entre perfiles similares con ingresos >50K.

--------------------------------------------------------------------------------


<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

### **Pureba con perfiles simulados**

Se probaron perfiles simulados para validar el comportamiento del sistema de recomendación.  
Para cada perfil, el sistema identifica usuarios reales con ingresos >50K que presentan mayor similitud y, a partir de sus características, genera recomendaciones concretas relacionadas con educación, ocupación y horas trabajadas.

</div>

<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

### **Conclusión final del modelo**

En este proyecto se ha desarrollado un sistema de recomendación basado en similitud entre usuarios, utilizando características demográficas y socioeconómicas para identificar patrones asociados a mayores niveles de ingresos.

A partir de la comparación entre usuarios con ingresos <=50K y perfiles similares con ingresos >50K, se han generado recomendaciones concretas orientadas a la mejora del perfil del usuario, como el aumento del nivel educativo, cambios en la ocupación o incremento de las horas trabajadas.

Este enfoque permite ofrecer recomendaciones interpretables y basadas en evidencia real del dataset, aunque presenta limitaciones al depender exclusivamente de similitudes y no considerar relaciones causales entre variables.

Como trabajo futuro, se podría complementar este sistema con modelos de clasificación o simulación de probabilidades para estimar el impacto de posibles cambios en el perfil del usuario.

</div>